In [4]:
from langfuse import get_client
import os 
from dotenv import load_dotenv
load_dotenv()


LANGFUSE_PUBLIC_KEY="pk-lf-d52922a4-6f16-4fc5-98f0-6c20cffbeb17"
LANGFUSE_SECRET_KEY="sk-lf-e041023c-b500-439a-9dd8-f30935d5b9ad"
LANGFUSE_HOST="https://langfuse.kodosumi.io/"


langfuse = get_client(public_key=LANGFUSE_PUBLIC_KEY)

In [2]:
import pandas as pd
# helper function
def pydantic_list_to_dataframe(pydantic_list):
    """
    Convert a list of pydantic objects to a pandas dataframe.
    """
    data = []
    for item in pydantic_list:
        data.append(item.dict())
    return pd.DataFrame(data)

In [5]:
traces = langfuse.api.trace.list(limit=5)
# pydantic_list_to_dataframe(traces.data).head(1)
traces

ConnectError: [Errno 61] Connection refused

In [7]:
from neo4j import GraphDatabase
import sys
from neo4j.exceptions import ServiceUnavailable, AuthError

# Neo4j connection details
NEO4J_URI = "neo4j://10.197.12.211:7687"  # or neo4j://127.0.0.1:7687
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "password123"
NEO4J_DATABASE = "politicalmonitoring-2025-11-06t04-54-54"

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD)
)

with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run("MATCH (n) RETURN count(n) as count")
    node_count = result.single()["count"]
    
    result = session.run("MATCH ()-[r]->() RETURN count(r) as count")
    rel_count = result.single()["count"]
    
    print(f"📊 Database Statistics:")
    print(f"   Total Nodes: {node_count:,}")
    print(f"   Total Relationships: {rel_count:,}")
    print()
# Get host IP information
print("🌐 Network Configuration:")
print()
print("   For Docker containers to connect to this Neo4j instance,")
print("   Neo4j must be configured to accept connections from Docker.")
print()
print("   Configuration steps:")
print("   1. Edit neo4j.conf (usually in /etc/neo4j/ or Neo4j Desktop settings)")
print("   2. Set: dbms.default_listen_address=0.0.0.0")
print("   3. Restart Neo4j")
print()
print("   Then use one of these URIs from Docker:")
print("   - neo4j://host.docker.internal:7687 (Mac/Windows Docker Desktop)")
print("   - neo4j://172.17.0.1:7687 (Linux Docker default bridge)")
print("   - neo4j://<your-host-ip>:7687")
print()

📊 Database Statistics:
   Total Nodes: 4,773
   Total Relationships: 21,752

🌐 Network Configuration:

   For Docker containers to connect to this Neo4j instance,
   Neo4j must be configured to accept connections from Docker.

   Configuration steps:
   1. Edit neo4j.conf (usually in /etc/neo4j/ or Neo4j Desktop settings)
   2. Set: dbms.default_listen_address=0.0.0.0
   3. Restart Neo4j

   Then use one of these URIs from Docker:
   - neo4j://host.docker.internal:7687 (Mac/Windows Docker Desktop)
   - neo4j://172.17.0.1:7687 (Linux Docker default bridge)
   - neo4j://<your-host-ip>:7687



In [ ]:
#!/usr/bin/env python3
"""
Neo4j Connection Test Script - FOR HOST MACHINE
Run this directly on your local machine (not in Docker)
"""

import sys

try:
    from neo4j import GraphDatabase
    from neo4j.exceptions import ServiceUnavailable, AuthError
except ImportError:
    print("ERROR: neo4j package not installed")
    print("Please install it with: pip install neo4j")
    sys.exit(1)

# Neo4j connection details
NEO4J_URI = "neo4j://localhost:7687"  # or neo4j://127.0.0.1:7687
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "password123"
NEO4J_DATABASE = "politicalmonitoring-2025-11-06t04-54-54"

def test_connection():
    """Test the connection to Neo4j and get your host IP."""
    
    print("=" * 70)
    print("Neo4j Connection Test - HOST MACHINE")
    print("=" * 70)
    print()
    
    driver = None
    try:
        print(f"📡 Connecting to Neo4j locally...")
        print(f"   URI: {NEO4J_URI}")
        print(f"   Database: {NEO4J_DATABASE}")
        print()
        
        driver = GraphDatabase.driver(
            NEO4J_URI,
            auth=(NEO4J_USER, NEO4J_PASSWORD)
        )
        
        driver.verify_connectivity()
        print("✅ Connection successful!")
        print()
        
        # Get database info
        with driver.session(database=NEO4J_DATABASE) as session:
            result = session.run("MATCH (n) RETURN count(n) as count")
            node_count = result.single()["count"]
            
            result = session.run("MATCH ()-[r]->() RETURN count(r) as count")
            rel_count = result.single()["count"]
            
            print(f"📊 Database Statistics:")
            print(f"   Total Nodes: {node_count:,}")
            print(f"   Total Relationships: {rel_count:,}")
            print()
        
        # Get host IP information
        print("🌐 Network Configuration:")
        print()
        print("   For Docker containers to connect to this Neo4j instance,")
        print("   Neo4j must be configured to accept connections from Docker.")
        print()
        print("   Configuration steps:")
        print("   1. Edit neo4j.conf (usually in /etc/neo4j/ or Neo4j Desktop settings)")
        print("   2. Set: dbms.default_listen_address=0.0.0.0")
        print("   3. Restart Neo4j")
        print()
        print("   Then use one of these URIs from Docker:")
        print("   - neo4j://host.docker.internal:7687 (Mac/Windows Docker Desktop)")
        print("   - neo4j://172.17.0.1:7687 (Linux Docker default bridge)")
        print("   - neo4j://<your-host-ip>:7687")
        print()
        
        # Try to get local IP
        import socket
        try:
            hostname = socket.gethostname()
            local_ip = socket.gethostbyname(hostname)
            print(f"   Your hostname: {hostname}")
            print(f"   Your local IP: {local_ip}")
            print()
        except:
            pass
        
        print("=" * 70)
        print("✅ Neo4j is running and accessible locally")
        print("=" * 70)
        print()
        
        return True
        
    except ServiceUnavailable as e:
        print("=" * 70)
        print("❌ Connection Failed - Service Unavailable")
        print("=" * 70)
        print()
        print(f"Error: {str(e)}")
        print()
        print("Neo4j is not running or not accessible.")
        print("Please start Neo4j and try again.")
        print()
        return False
        
    except AuthError as e:
        print("=" * 70)
        print("❌ Connection Failed - Authentication Error")
        print("=" * 70)
        print()
        print(f"Error: {str(e)}")
        print()
        print("Username or password is incorrect.")
        print()
        return False
        
    except Exception as e:
        print("=" * 70)
        print("❌ Connection Failed")
        print("=" * 70)
        print()
        print(f"Error: {str(e)}")
        print(f"Error Type: {type(e).__name__}")
        print()
        return False
        
    finally:
        if driver:
            driver.close()

if __name__ == "__main__":
    result = test_connection()
    sys.exit(0 if result else 1)